In [2]:
import pandas as pd
import duckdb

data = [
    ["A", "2026-07-01 08:00:00", "NORMAL", 72],
    ["A", "2026-07-01 08:03:00", "ERROR", 88],
    ["A", "2026-07-01 08:05:00", "NORMAL", 76],

    ["B", "2026-07-01 08:01:00", "NORMAL", 70],
    ["B", "2026-07-01 08:04:00", "ERROR", 91],
    ["B", "2026-07-01 08:06:00", "ERROR", 95],

    ["C", "2026-07-01 08:02:00", "NORMAL", 68],
    ["C", "2026-07-01 08:07:00", "NORMAL", 73],
]

df = pd.DataFrame(
    data,
    columns=["device_id", "collect_time", "status", "temp_value"]
)

df["collect_time"] = pd.to_datetime(df["collect_time"])

print(df)


  device_id        collect_time  status  temp_value
0         A 2026-07-01 08:00:00  NORMAL          72
1         A 2026-07-01 08:03:00   ERROR          88
2         A 2026-07-01 08:05:00  NORMAL          76
3         B 2026-07-01 08:01:00  NORMAL          70
4         B 2026-07-01 08:04:00   ERROR          91
5         B 2026-07-01 08:06:00   ERROR          95
6         C 2026-07-01 08:02:00  NORMAL          68
7         C 2026-07-01 08:07:00  NORMAL          73


# 题目要求

## 分别使用 SQL 和 Pandas 完成：

* **找出每个设备的最新一条状态记录。**

* **最终输出：**

|device_id | collect_time          | status | temp_value|
|----------|-----------------------|--------|-----------|
|A         | 2026-07-01 08:05:00   | NORMAL | 76|
|B         | 2026-07-01 08:06:00   | ERROR  | 95|
|C         | 2026-07-01 08:07:00   | NORMAL | 73|

* **SQL 方向：** 使用 ROW_NUMBER()。

* **Pandas 方向：** 可以用排序后去重，或者 groupby().rank()。

In [27]:
# SQL轨道

query = """
WITH temp_value_rank AS(
SELECT
    device_id,
    collect_time,
    status,
    temp_value,
    ROW_NUMBER() OVER(PARTITION BY device_id ORDER BY collect_time DESC) AS rn
FROM df 
)

SELECT
    device_id,
    collect_time,
    status,
    temp_value,
FROM temp_value_rank
WHERE rn = 1
ORDER BY device_id

"""

df_sql = duckdb.execute(query).fetchdf()
df_sql

,device_id,collect_time,status,temp_value
0,A,2026-07-01 08:05:00,NORMAL,76
1,B,2026-07-01 08:06:00,ERROR,95
2,C,2026-07-01 08:07:00,NORMAL,73


In [28]:
# pandas轨道
df = df.sort_values(['device_id','collect_time'],ascending=[True,False])
df['rn'] = df.groupby('device_id')['collect_time'].cumcount()+1

df_pd = (
    df
    .loc[df['rn'] == 1]
    .drop(columns='rn')
    .reset_index(drop=True)
)

df_pd

,device_id,collect_time,status,temp_value
0,A,2026-07-01 08:05:00,NORMAL,76
1,B,2026-07-01 08:06:00,ERROR,95
2,C,2026-07-01 08:07:00,NORMAL,73
